In [21]:
import dotenv
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()

OUTPUT_DIR = "./export_supabase/vtc_test"
# OUTPUT_DIR_Synt = "./synthetic_csv"

In [7]:
driver_reviews = pd.read_csv(f"{OUTPUT_DIR}/driver_reviews.csv")
locations = pd.read_csv(f"{OUTPUT_DIR}/locations.csv")
payments = pd.read_csv(f"{OUTPUT_DIR}/payments.csv")
promotions = pd.read_csv(f"{OUTPUT_DIR}/promotions.csv")
rides = pd.read_csv(f"{OUTPUT_DIR}/rides.csv")
users = pd.read_csv(f"{OUTPUT_DIR}/users.csv")
vehicles = pd.read_csv(f"{OUTPUT_DIR}/vehicles.csv")

print("Lenght driver_reviews:", len(driver_reviews))
print("Lenght locations:", len(locations))
print("Lenght payments:", len(payments))
print("Lenght promotions:", len(promotions))
print("Lenght rides:", len(rides))
print("Lenght users:", len(users))
print("Lenght vehicles:", len(vehicles))

Lenght driver_reviews: 2
Lenght locations: 2
Lenght payments: 2
Lenght promotions: 2
Lenght rides: 2
Lenght users: 4
Lenght vehicles: 2


In [8]:
driver_reviews.head()

,id,driver_id,customer_id,rating,comment,review_date
0,1,2,1,5.0,"Great ride, very friendly driver!",2026-04-20 20:08:45.897727+00:00
1,2,4,3,4.5,"Smooth ride, but the car was a bit messy.",2026-04-20 20:08:45.897727+00:00


In [9]:
locations.head()

,id,address,latitude,longitude
0,1,"123 Main St, Anytown, USA",40.7128,-74.0060
1,2,"456 Elm St, Othertown, USA",34.0522,-118.2437


In [10]:
payments.head()

,id,ride_id,amount,payment_method,payment_time,transaction_id
0,1,1,25.0,credit card,2026-04-20 20:08:45.897727+00:00,TXN123456
1,2,2,30.0,paypal,2026-04-20 20:08:45.897727+00:00,TXN789012


In [11]:
promotions.head()

,id,code,description,discount_percentage,valid_from,valid_to
0,1,WELCOME10,10% off for new users,10.0,2023-09-01,2023-12-31
1,2,SAVE20,20% off on rides over $50,20.0,2023-10-01,2023-11-30


In [12]:
rides.head()

,id,customer_id,driver_id,vehicle_id,pickup_location_id,dropoff_location_id,start_time,end_time,status,distance,fare
0,1,1,2,1,1,2,2023-10-01 08:00:00+00:00,2023-10-01 08:30:00+00:00,completed,15.5,25.0
1,2,3,4,2,2,1,2023-10-02 09:00:00+00:00,2023-10-02 09:45:00+00:00,completed,20.0,30.0


In [13]:
users.head()

,id,name,email,phone_number,user_type,created_at,rating,profile_picture
0,1,Alice Johnson,alice@example.com,123-456-7890,customer,2026-04-20 20:08:45.897727+00:00,4.5,alice.jpg
1,2,Bob Smith,bob@example.com,234-567-8901,driver,2026-04-20 20:08:45.897727+00:00,4.8,bob.jpg
2,3,Charlie Brown,charlie@example.com,345-678-9012,customer,2026-04-20 20:08:45.897727+00:00,4.2,charlie.jpg
3,4,David Wilson,david@example.com,456-789-0123,driver,2026-04-20 20:08:45.897727+00:00,4.9,david.jpg


In [14]:
vehicles.head()

,id,driver_id,make,model,year,license_plate,color,insurance_expiry
0,1,2,Toyota,Corolla,2018,ABC123,Blue,2024-05-01
1,2,4,Honda,Civic,2020,XYZ789,Red,2025-08-15


## Creamos información random

In [22]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

np.random.seed(42)
random.seed(42)
# 1) users
n_users = 100_000
n_drivers = 10_000

user_ids = np.arange(1, n_users + 1)
user_types = np.array(['customer'] * n_users)
user_types[:n_drivers] = 'driver'
np.random.shuffle(user_types)

users = pd.DataFrame({
    "id": user_ids,
    "name": [f"User {i}" for i in user_ids],
    "email": [f"user{i}@example.com" for i in user_ids],
    "phone_number": [f"{1000000000 + i}" for i in user_ids],
    "user_type": user_types,
    "created_at": pd.to_datetime("2023-01-01") + pd.to_timedelta(np.random.randint(0, 365, n_users), unit="D"),
    "rating": np.round(3.5 + np.random.rand(n_users) * 1.5, 1),
    "profile_picture": [f"user{i}.jpg" for i in user_ids],
})
users.to_csv(f"{OUTPUT_DIR}/users.csv", index=False)

# 2) vehicles (uno por driver)
drivers = users[users["user_type"] == "driver"].copy()
drivers = drivers.sort_values("id").reset_index(drop=True)
n_vehicles = len(drivers)

makes = ['Toyota','Honda','Ford','Hyundai','Kia','Volkswagen','Seat','Renault']
models = ['Corolla','Civic','Focus','i30','Ceed','Golf','Ibiza','Clio']
colors = ['Blue','Red','Black','White','Silver','Grey']

vehicles = pd.DataFrame({
    "id": np.arange(1, n_vehicles + 1),
    "driver_id": drivers["id"].values,
    "make": np.random.choice(makes, n_vehicles),
    "model": np.random.choice(models, n_vehicles),
    "year": np.random.randint(2015, 2023, n_vehicles),
    "license_plate": [f"PLT{1000 + i:04d}" for i in range(n_vehicles)],
    "color": np.random.choice(colors, n_vehicles),
    "insurance_expiry": pd.to_datetime("2024-01-01") + pd.to_timedelta(np.random.randint(0, 365, n_vehicles), unit="D"),
})
vehicles.to_csv(f"{OUTPUT_DIR}/vehicles.csv", index=False)

# 3) locations
n_locations = 20_000
locations = pd.DataFrame({
    "id": np.arange(1, n_locations + 1),
    "address": [f"Calle {i}, Ciudad {i % 50 + 1}" for i in range(1, n_locations + 1)],
    "latitude": 40.0 + np.random.rand(n_locations) * 0.5,
    "longitude": -3.8 + np.random.rand(n_locations) * 0.5,
})
locations.to_csv(f"{OUTPUT_DIR}/locations.csv", index=False)

# 4) promotions
n_promos = 20
discounts = [5.0, 10.0, 15.0, 20.0]
promotions = pd.DataFrame({
    "id": np.arange(1, n_promos + 1),
    "code": [f"PROMO{i:03d}" for i in range(1, n_promos + 1)],
    "description": [f"Promo auto {i}" for i in range(1, n_promos + 1)],
    "discount_percentage": np.random.choice(discounts, n_promos),
    "valid_from": pd.to_datetime("2023-01-01") + pd.to_timedelta(np.random.randint(0, 180, n_promos), unit="D"),
    "valid_to": pd.to_datetime("2023-07-01") + pd.to_timedelta(np.random.randint(0, 180, n_promos), unit="D"),
})
promotions.to_csv(f"{OUTPUT_DIR}/promotions.csv", index=False)

# 5) rides
n_rides = 500_000
customers = users[users["user_type"] == "customer"].copy()

def sample_times(n):
    base_dates = pd.to_datetime("2023-09-01") + pd.to_timedelta(np.random.randint(0, 90, n), unit="D")
    peak_hours = np.array([7,8,9,17,18,19,20])
    off_hours = np.array([10,11,12,13,14,15,16,21,22,23])
    hours = []
    for _ in range(n):
        if random.random() < 0.6:
            hours.append(int(np.random.choice(peak_hours)))
        else:
            hours.append(int(np.random.choice(off_hours)))
    minutes = np.random.randint(0, 60, n)
    start = base_dates + pd.to_timedelta(hours, unit="h") + pd.to_timedelta(minutes, unit="m")
    durations = np.random.randint(10, 50, n)  # minutos
    end = start + pd.to_timedelta(durations, unit="m")
    return start, end

ride_start, ride_end = sample_times(n_rides)

pickup_ids = np.random.randint(1, n_locations + 1, n_rides)
dropoff_ids = np.random.randint(1, n_locations + 1, n_rides)
same = pickup_ids == dropoff_ids
dropoff_ids[same] = (dropoff_ids[same] % n_locations) + 1

driver_ids = np.random.choice(drivers["id"].values, n_rides)
vehicle_map = vehicles.set_index("driver_id")["id"].to_dict()
vehicle_ids = [vehicle_map[d] for d in driver_ids]

distance = np.round(2.0 + np.random.rand(n_rides) * 23.0, 1)
fare = np.round(2.0 + distance * (0.7 + np.random.rand(n_rides) * 0.6) + np.random.rand(n_rides) * 3.0, 2)

rides = pd.DataFrame({
    "id": np.arange(1, n_rides + 1),
    "customer_id": np.random.choice(customers["id"].values, n_rides),
    "driver_id": driver_ids,
    "vehicle_id": vehicle_ids,
    "pickup_location_id": pickup_ids,
    "dropoff_location_id": dropoff_ids,
    "start_time": ride_start,
    "end_time": ride_end,
    "status": "completed",
    "distance": distance,
    "fare": fare,
})
rides.to_csv(f"{OUTPUT_DIR}/rides.csv", index=False)

# 6) payments
methods = ["credit card", "paypal", "cash"]
payments = pd.DataFrame({
    "id": np.arange(1, n_rides + 1),
    "ride_id": rides["id"],
    "amount": rides["fare"],
    "payment_method": np.random.choice(methods, n_rides, p=[0.6, 0.25, 0.15]),
    "payment_time": rides["end_time"] + pd.to_timedelta(np.random.randint(0, 10, n_rides), unit="m"),
    "transaction_id": [f"TXN{i:08d}" for i in rides["id"]],
})
payments.to_csv(f"{OUTPUT_DIR}/payments.csv", index=False)

# 7) driver_reviews (~20% de rides)
mask = np.random.rand(n_rides) < 0.2
review_rides = rides[mask].copy()
n_reviews = len(review_rides)

comments = [
    "Great ride, very friendly driver!",
    "Smooth ride, car was clean.",
    "Driver arrived on time.",
    "Good service, would ride again.",
    "Ride was okay, a bit of traffic.",
    "Excellent experience, highly recommended.",
    "Driver was polite and professional."
]

driver_reviews = pd.DataFrame({
    "id": np.arange(1, n_reviews + 1),
    "driver_id": review_rides["driver_id"].values,
    "customer_id": review_rides["customer_id"].values,
    "rating": np.round(3.5 + np.random.rand(n_reviews) * 1.5, 1),
    "comment": np.random.choice(comments, n_reviews),
    "review_date": review_rides["end_time"].values + np.random.randint(0, 120, n_reviews).astype("timedelta64[m]"),
})
driver_reviews.to_csv(f"{OUTPUT_DIR}/driver_reviews.csv", index=False)


In [23]:
driver_reviews = pd.read_csv(f"{OUTPUT_DIR}/driver_reviews.csv")
locations = pd.read_csv(f"{OUTPUT_DIR}/locations.csv")
payments = pd.read_csv(f"{OUTPUT_DIR}/payments.csv")
promotions = pd.read_csv(f"{OUTPUT_DIR}/promotions.csv")
rides = pd.read_csv(f"{OUTPUT_DIR}/rides.csv")
users = pd.read_csv(f"{OUTPUT_DIR}/users.csv")
vehicles = pd.read_csv(f"{OUTPUT_DIR}/vehicles.csv")

print("Lenght driver_reviews:", len(driver_reviews))
print("Lenght locations:", len(locations))
print("Lenght payments:", len(payments))
print("Lenght promotions:", len(promotions))
print("Lenght rides:", len(rides))
print("Lenght users:", len(users))
print("Lenght vehicles:", len(vehicles))

Lenght driver_reviews: 100320
Lenght locations: 20000
Lenght payments: 500000
Lenght promotions: 20
Lenght rides: 500000
Lenght users: 100000
Lenght vehicles: 10000


In [ ]:
host = os.getenv("HOST"),
port = os.getenv("PORT"),
database = os.getenv("DB"),
user = os.getenv("USER"),
password = os.getenv("PASSWORD")

In [34]:
len(driver_reviews)

100320

In [35]:
driver_reviews.head()

,id,driver_id,customer_id,rating,comment,review_date
0,1,26424,66703,3.6,Driver arrived on time.,2023-11-26 22:55:00
1,2,71992,54223,4.8,"Ride was okay, a bit of traffic.",2023-11-12 17:31:00
2,3,71767,90599,4.2,"Smooth ride, car was clean.",2023-10-10 14:38:00
3,4,67711,8385,4.2,Driver arrived on time.,2023-09-24 07:45:00
4,5,80349,45760,3.8,Driver arrived on time.,2023-10-30 15:46:00


In [36]:
len(driver_reviews['driver_id'].unique())

9999

In [48]:
## Subir las tablas a supabase

conn = psycopg2.connect(
    host= os.getenv("HOST"),
    port= os.getenv("PORT"),
    database= os.getenv("DB"),
    user= os.getenv("USER"),
    password= os.getenv("PASSWORD"),
    connect_timeout = 0, ## Desactivamos el timeout para conexiones largas
    options = '-c statement_timeout=0' ## Desactivamos el timeout para consultas largas
)

cur = conn.cursor()

with open(f"{OUTPUT_DIR}/rides.csv", "r", encoding="utf-8") as f:
    cur.copy_expert("COPY public.rides FROM STDIN WITH CSV HEADER", f)

conn.commit()
cur.close()
conn.close()


QueryCanceled: canceling statement due to statement timeout
CONTEXT:  COPY rides, line 332299: "332298,29209,13147,1324,12971,456,2023-11-08 12:42:00,2023-11-08 13:06:00,completed,15.0,19.94"


In [ ]:
##Subir archivos grandes dividiendolos en partes
INPUT = f"{OUTPUT_DIR}/rides.csv"
CHUNK_SIZE = 50000

#Troceamos el archivo en partes de 50k filas y las guardamos en el mismo directorio
i = 0
for chunk in pd.read_csv(INPUT, chunksize=CHUNK_SIZE):
    chunk.to_csv(f"{OUTPUT_DIR}/rides_part_{i}.csv", index=False)
    print(f"Generado rides_part_{i}.csv")
    i += 1



Generado rides_part_0.csv
Generado rides_part_1.csv
Generado rides_part_2.csv
Generado rides_part_3.csv
Generado rides_part_4.csv
Generado rides_part_5.csv
Generado rides_part_6.csv
Generado rides_part_7.csv
Generado rides_part_8.csv
Generado rides_part_9.csv
